# Лабораторная работа №1. Автоматизированный сбор данных. Web-scraping

**Дисциплина:** Новые технологии в РПС  
**Вариант:** 9 (Сбор отзывов с сервиса *Otzovik* по 5 категориям рейтинга 1–5 звёзд)  
**Выполнил:** студент группы ...

## 1. Описание задачи
Цель работы — разработать автоматизированный скрипт для сбора отзывов по выбранному объекту с сервиса «Отзовик».
Требования варианта 9:
- Сбор отзывов для каждого количества звёзд (от 1 до 5);
- Программное создание директорий `dataset/1/` ... `dataset/5/` (ручное создание запрещено);
- Форматирование имен файлов с ведущими нулями (`0000.txt`, `0001.txt`, ...) через строковые методы Python;
- Исключение дублирования данных;
- Поддержка Уровня 1 (краткий отзыв) и Уровня 2 (полный текст отзыва);
- Соответствие стандартам оформления PEP8.

## 2. Установка зависимостей и импорт библиотек

In [ ]:
# Раскомментируйте при запуске в Google Colab:
# !pip install requests beautifulsoup4 lxml matplotlib

from pathlib import Path
import random
import re
import time
from typing import Dict, List, Optional, Set
from urllib.parse import urljoin

import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup

print("Все библиотеки успешно импортированы!")

## 3. Модуль управления датасетом (DatasetManager)
Класс отвечает за создание каталогов, дедупликацию и сохранение файлов с именами вида `0000.txt` с помощью метода `str.zfill(4)`.

In [ ]:
class DatasetManager:
    """Управление папками датасета, нумерацией и исключением дубликатов."""

    def __init__(self, base_dir: str = "dataset", target_per_class: int = 10) -> None:
        self.base_dir = Path(base_dir)
        self.target_per_class = target_per_class
        self.categories: List[int] = [1, 2, 3, 4, 5]
        self._seen_ids: Set[str] = set()
        self._counts: Dict[int, int] = {cat: 0 for cat in self.categories}

        self._init_folders()
        self._sync_with_disk()

    def _init_folders(self) -> None:
        self.base_dir.mkdir(parents=True, exist_ok=True)
        for cat in self.categories:
            (self.base_dir / str(cat)).mkdir(parents=True, exist_ok=True)

    def _sync_with_disk(self) -> None:
        for cat in self.categories:
            folder = self.base_dir / str(cat)
            if folder.exists():
                self._counts[cat] = len(list(folder.glob("*.txt")))

    def is_quota_filled(self, category: int) -> bool:
        return self._counts.get(category, 0) >= self.target_per_class

    def is_all_completed(self) -> bool:
        return all(self.is_quota_filled(cat) for cat in self.categories)

    def save_review(self, category: int, review_id: str, title: str, text: str) -> bool:
        if category not in self.categories or self.is_quota_filled(category):
            return False
        if review_id in self._seen_ids:
            return False

        idx = self._counts[category]
        filename = f"{str(idx).zfill(4)}.txt"
        filepath = self.base_dir / str(category) / filename

        payload = f"{title.strip()}\n\n{text.strip()}\n"
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(payload)

        self._seen_ids.add(review_id)
        self._counts[category] += 1
        return True

    def get_stats(self) -> Dict[int, int]:
        return self._counts.copy()

print("Класс DatasetManager объявлен!")

## 4. Модуль сетевого взаимодействия и скрейпинга (OtzovikScraper)
Реализует обход страниц пагинации, симуляцию браузера и парсинг оценок с текстами.

In [ ]:
class OtzovikScraper:
    """Клиент для обхода страниц и парсинга отзывов с Otzovik."""

    BASE_URL = "https://otzovik.com"

    def __init__(self, object_slug: str = "sberbank_rossii", dataset_manager: Optional[DatasetManager] = None, full_text: bool = False) -> None:
        self.object_slug = object_slug
        self.manager = dataset_manager or DatasetManager()
        self.full_text = full_text
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7",
            "Referer": "https://otzovik.com/",
        })

    def _fetch_html(self, url: str) -> Optional[str]:
        try:
            resp = self.session.get(url, timeout=12)
            if resp.status_code == 200:
                return resp.text
        except requests.RequestException as err:
            print(f"Ошибка сети: {err}")
        return None

    def _parse_rating(self, card: BeautifulSoup) -> Optional[int]:
        pattern = re.compile(r"Общий рейтинг:\s*(\d+)")
        score_tag = card.find(attrs={"title": pattern})
        if score_tag:
            match = pattern.search(score_tag["title"])
            if match:
                return int(match.group(1))
        mapping = {"Ужасно": 1, "Плохо": 2, "Средне": 3, "Хорошо": 4, "Отлично": 5}
        for word, val in mapping.items():
            if card.find(attrs={"title": re.compile(rf"\b{word}\b", re.I)}):
                return val
        return None

    def parse_page(self, page_num: int) -> int:
        url = f"{self.BASE_URL}/reviews/{self.object_slug}/" if page_num == 1 else f"{self.BASE_URL}/reviews/{self.object_slug}/{page_num}/"
        html = self._fetch_html(url)
        if not html:
            return 0

        soup = BeautifulSoup(html, "lxml")
        saved = 0
        for r_link in soup.find_all("a", href=re.compile(r"/review_\d+\.html")):
            href = r_link.get("href", "")
            id_match = re.search(r"/review_(\d+)\.html", href)
            if not id_match:
                continue
            review_id = id_match.group(1)
            card = r_link.find_parent("div", class_=lambda c: c and "item" in c.split()) or r_link.find_parent("div")
            if not card:
                continue

            rating = self._parse_rating(card)
            if not rating or self.manager.is_quota_filled(rating):
                continue

            title = r_link.get_text(strip=True)
            body_el = card.find(class_=re.compile(r"review-body|description|review-snip"))
            text = body_el.get_text(" ", strip=True) if body_el else title

            if self.manager.save_review(rating, review_id, title, text):
                saved += 1
        return saved

print("Класс OtzovikScraper объявлен!")

## 5. Запуск сбора данных и визуализация датасета

In [ ]:
manager = DatasetManager(base_dir="dataset", target_per_class=15)
scraper = OtzovikScraper(object_slug="sberbank_rossii", dataset_manager=manager, full_text=False)

# Сбор с первых страниц
for p in range(1, 4):
    if manager.is_all_completed():
        break
    saved = scraper.parse_page(p)
    print(f"Страница {p}: сохранено {saved} отзывов. Текущее состояние: {manager.get_stats()}")
    time.sleep(2.0)

# Визуализация распределения классов
stats = manager.get_stats()
categories = [f"{k} звёзд" for k in stats.keys()]
counts = list(stats.values())

plt.figure(figsize=(8, 4))
bars = plt.bar(categories, counts, color=["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#27ae60"])
plt.title("Распределение собранных отзывов по классам рейтинга")
plt.xlabel("Оценка")
plt.ylabel("Количество файлов")
plt.grid(axis="y", linestyle="--", alpha=0.7)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.2, int(yval), ha="center", va="bottom")
plt.show()

## 6. Проверка сохраненных файлов
Выведем образец содержимого файла отзыва с ведущими нулями.

In [ ]:
sample_file = Path("dataset/1/0000.txt")
if sample_file.exists():
    print(f"--- Пример файла {sample_file} ---")
    print(sample_file.read_text(encoding="utf-8"))
else:
    print("Файл пока не создан. Запустите сбор выше.")